# Topic Modeling
## This notebook outlines the concepts involved in Topic Modeling


Topic modeling is a statistical model to **discover** the abstract "topics" that occur in a collection of documents

It is commonly used in text document. But nowadays, in social media analysis, topic modeling is an emerging research area.

One of the most popular algorithms used is **Latent Dirichlet Allocation** which was proposed by
David Blei et al in 2003.

Dataset:
https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Tokenize
    - Stop words removal
    - Non-alphabetic words removal
    - Lowercase
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Visualize the topics

### Install the necessary library

In [1]:
%%capture
! pip install gensim

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Import the necessary libraries

In [3]:
from pprint import pprint

import pandas as pd

import gensim
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.corpora import Dictionary

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Download the dataset

In [4]:
!wget https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import urllib.request

url = "https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv"
urllib.request.urlretrieve(url, "kaggledatasets.csv")

print("Downloaded!")

Downloaded!


### Load the dataset

In [6]:
df = pd.read_csv("kaggledatasets.csv")
df.head()

,Title,Subtitle,Owner,Votes,Versions,Tags,Data Type,Size,License,Views,Download,Kernels,Topics,URL,Description
0,Credit Card Fraud Detection,Anonymized credit card transactions labeled as...,Machine Learning Group - ULB,1241,"Version 2,2016-11-05|Version 1,2016-11-03",crime\nfinance,CSV,144 MB,ODbL,"442,136 views","53,128 downloads","1,782 kernels",26 topics,https://www.kaggle.com/mlg-ulb/creditcardfraud,The datasets contains transactions made by cre...
1,European Soccer Database,"25k+ matches, players & teams attributes for E...",Hugo Mathien,1046,"Version 10,2016-10-24|Version 9,2016-10-24|Ver...",association football\neurope,SQLite,299 MB,ODbL,"396,214 views","46,367 downloads","1,459 kernels",75 topics,https://www.kaggle.com/hugomathien/soccer,The ultimate Soccer database for data analysis...
2,TMDB 5000 Movie Dataset,"Metadata on ~5,000 movies from TMDb",The Movie Database (TMDb),1024,"Version 2,2017-09-28",film,CSV,44 MB,Other,"446,255 views","62,002 downloads","1,394 kernels",46 topics,https://www.kaggle.com/tmdb/tmdb-movie-metadata,Background\nWhat can we say about the success ...
3,Global Terrorism Database,"More than 170,000 terrorist attacks worldwide,...",START Consortium,789,"Version 2,2017-07-19|Version 1,2016-12-08",crime\nterrorism\ninternational relations,CSV,144 MB,Other,"187,877 views","26,309 downloads",608 kernels,11 topics,https://www.kaggle.com/START-UMD/gtd,"Context\nInformation on more than 170,000 Terr..."
4,Bitcoin Historical Data,Bitcoin data at 1-min intervals from select ex...,Zielak,618,"Version 11,2018-01-11|Version 10,2017-11-17|Ve...",history\nfinance,CSV,119 MB,CC4,"146,734 views","16,868 downloads",68 kernels,13 topics,https://www.kaggle.com/mczielinski/bitcoin-his...,Context\nBitcoin is the longest running and mo...


### Explore the dataset

In [7]:
df.columns

Index(['Title', 'Subtitle', 'Owner', 'Votes', 'Versions', 'Tags', 'Data Type',
       'Size', 'License', 'Views', 'Download', 'Kernels', 'Topics', 'URL',
       'Description'],
      dtype='object')

In [8]:
df.dtypes

Title          object
Subtitle       object
Owner          object
Votes           int64
Versions       object
Tags           object
Data Type      object
Size           object
License        object
Views          object
Download       object
Kernels        object
Topics         object
URL            object
Description    object
dtype: object

In [9]:
df['Description'].iloc[0]

"The datasets contains transactions made by credit cards in September 2013 by european cardholders. This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.\nIt contains only numerical input variables which are the result of a PCA transformation. Unfortunately, due to confidentiality issues, we cannot provide the original features and more background information about the data. Features V1, V2, ... V28 are the principal components obtained with PCA, the only features which have not been transformed with PCA are 'Time' and 'Amount'. Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset. The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-senstive learning. Feature 'Class' is the response variable and it takes value 1 in case o

In [10]:
df.isnull().sum()

Title            0
Subtitle       104
Owner            0
Votes            0
Versions         5
Tags           542
Data Type        0
Size             0
License          0
Views            5
Download        15
Kernels        944
Topics         592
URL              0
Description      5
dtype: int64

### Extract the data for topic modeling

In [14]:
for i in df['Description'].head().items():
    raw = str(i[1]).lower()
    print(raw)

the datasets contains transactions made by credit cards in september 2013 by european cardholders. this dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. the dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.
it contains only numerical input variables which are the result of a pca transformation. unfortunately, due to confidentiality issues, we cannot provide the original features and more background information about the data. features v1, v2, ... v28 are the principal components obtained with pca, the only features which have not been transformed with pca are 'time' and 'amount'. feature 'time' contains the seconds elapsed between each transaction and the first transaction in the dataset. the feature 'amount' is the transaction amount, this feature can be used for example-dependant cost-senstive learning. feature 'class' is the response variable and it takes value 1 in case of 

In [15]:
texts = []

for idx, desc in df['Description'].items():
    raw = str(desc).lower()
    texts.append(raw)

### Pre-process the dataset
- Tokenize
- Stop words removal
- Non-alphabetic words removal
- Lowercase
- Define them

### Define the pattern, tokenizer, stop words and lemmatizer

In [16]:
pattern = r'\b[^\d\W]+\b'
tokenizer = RegexpTokenizer(pattern)
en_stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Preprocess

In [17]:
texts = []

for idx, desc in df['Description'].items():
    # clean and tokenize document string
    raw_text = str(desc).lower()
    tokens = tokenizer.tokenize(raw_text)

    # remove stop words
    stopped_tokens = [t for t in tokens if t not in en_stop]

    # lemmatize tokens
    lemma_tokens = [lemmatizer.lemmatize(t) for t in stopped_tokens]

    # remove single-character words
    filtered_tokens = [t for t in lemma_tokens if len(t) > 1]

    # add tokens to list
    texts.append(filtered_tokens)

# check one example
print(texts[0])

['datasets', 'contains', 'transaction', 'made', 'credit', 'card', 'september', 'european', 'cardholder', 'dataset', 'present', 'transaction', 'occurred', 'two', 'day', 'fraud', 'transaction', 'dataset', 'highly', 'unbalanced', 'positive', 'class', 'fraud', 'account', 'transaction', 'contains', 'numerical', 'input', 'variable', 'result', 'pca', 'transformation', 'unfortunately', 'due', 'confidentiality', 'issue', 'cannot', 'provide', 'original', 'feature', 'background', 'information', 'data', 'feature', 'principal', 'component', 'obtained', 'pca', 'feature', 'transformed', 'pca', 'time', 'amount', 'feature', 'time', 'contains', 'second', 'elapsed', 'transaction', 'first', 'transaction', 'dataset', 'feature', 'amount', 'transaction', 'amount', 'feature', 'used', 'example', 'dependant', 'cost', 'senstive', 'learning', 'feature', 'class', 'response', 'variable', 'take', 'value', 'case', 'fraud', 'otherwise', 'given', 'class', 'imbalance', 'ratio', 'recommend', 'measuring', 'accuracy', 'usi

### Create a dictionary

In [18]:
dictionary = Dictionary(texts)
dictionary[1]

'account'

In [23]:
dictionary.token2id["group"]

54

### Filter low frequency words

In [25]:
dictionary.filter_extremes(no_below=10, no_above=0.5)
# convert tokenized documents into a document-term matrix
corpus = [dictionary.doc2bow(text) for text in texts]
print(corpus[:5])

[[(0, 3), (1, 1), (2, 2), (3, 3), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 3), (13, 2), (14, 1), (15, 1), (16, 1), (17, 1), (18, 3), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 7), (31, 1), (32, 4), (33, 1), (34, 1), (35, 1), (36, 3), (37, 1), (38, 1), (39, 1), (40, 1), (41, 1), (42, 2), (43, 1), (44, 1), (45, 1), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 1), (58, 1), (59, 1), (60, 1), (61, 1), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 1), (72, 1), (73, 1), (74, 2), (75, 1), (76, 7), (77, 1), (78, 1), (79, 1), (80, 1), (81, 1), (82, 1), (83, 1), (84, 2)], [(9, 1), (10, 1), (12, 2), (23, 1), (28, 1), (30, 1), (31, 1), (36, 3), (42, 1), (43, 1), (52, 2), (55, 3), (57, 1), (60, 2), (61, 1), (74, 2), (82, 2), (83, 1), (85, 2), (86, 2), (87, 1), (88, 1), (89, 1), (90, 1), (91, 3), (92, 1), (93, 1),

### Create an index to word dictionary

In [27]:
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

### Train the Topic model

### What is LDA (Latent Dirichlet Allocation)?

**LDA** is a topic modeling algorithm that automatically discovers **topics** hidden in a collection of documents.


- Each **document** is made up of several **topics**.  
- Each **topic** is made up of certain **words** that often appear together.

Example:  
A document might be **70% "machine learning"** and **30% "healthcare"**,  
and the "machine learning" topic might include words like *model, data, algorithm*.


### How it works (in plain terms)

1. **Start with random guesses** of which topics belong to which words.  
2. For each word, update the topic assignment based on:
   - how common the topic is in the document  
   - how common the word is in that topic  
3. Repeat many times.  
4. Gradually, the model finds stable patterns where topics and words fit well.


**In short:**  
> LDA learns which words group together into topics  
> and how much of each topic is present in every document.


In [31]:
ldamodel = LdaModel(corpus, num_topics=15, id2word = id2word, passes=20)

### Display the topics

In [30]:
pprint(ldamodel.top_topics(corpus,topn=5))

[([(np.float32(0.009869711), 'csv'),
   (np.float32(0.009556932), 'file'),
   (np.float32(0.009180266), 'information'),
   (np.float32(0.006754328), 'http'),
   (np.float32(0.006315075), 'contains')],
  np.float64(-0.7557210659182668)),
 ([(np.float32(0.038099453), 'player'),
   (np.float32(0.033612337), 'game'),
   (np.float32(0.025942868), 'team'),
   (np.float32(0.017601488), 'match'),
   (np.float32(0.012591612), 'number')],
  np.float64(-0.994332548051603)),
 ([(np.float32(0.045771107), 'model'),
   (np.float32(0.044634696), 'image'),
   (np.float32(0.03420456), 'trained'),
   (np.float32(0.023576856), 'pre'),
   (np.float32(0.020881431), 'feature')],
  np.float64(-1.2854723052038737)),
 ([(np.float32(0.022577794), 'survey'),
   (np.float32(0.01957696), 'age'),
   (np.float32(0.017122876), 'education'),
   (np.float32(0.0144620985), 'health'),
   (np.float32(0.014208182), 'http')],
  np.float64(-1.3037541580479926)),
 ([(np.float32(0.029594941), 'day'),
   (np.float32(0.02626687),

### Display the 15 topics with words

In [35]:
for idx in range(15):
    print("Topic #%s:" % idx, ldamodel.print_topic(idx, 10))

Topic #0: 0.060*"model" + 0.048*"trained" + 0.033*"pre" + 0.026*"feature" + 0.018*"network" + 0.017*"image" + 0.014*"accuracy" + 0.013*"large" + 0.013*"architecture" + 0.013*"depth"
Topic #1: 0.043*"question" + 0.039*"score" + 0.038*"student" + 0.032*"team" + 0.032*"player" + 0.031*"school" + 0.022*"answer" + 0.021*"percentage" + 0.019*"set" + 0.019*"shot"
Topic #2: 0.031*"cell" + 0.031*"instance" + 0.020*"weapon" + 0.019*"attribute" + 0.017*"group" + 0.017*"time" + 0.014*"solar" + 0.013*"number" + 0.011*"type" + 0.011*"size"
Topic #3: 0.069*"csv" + 0.027*"numeric" + 0.023*"file" + 0.020*"government" + 0.018*"job" + 0.017*"txt" + 0.016*"federal" + 0.016*"election" + 0.014*"trump" + 0.014*"member"
Topic #4: 0.014*"product" + 0.012*"sale" + 0.011*"row" + 0.011*"date" + 0.011*"csv" + 0.010*"song" + 0.010*"new" + 0.010*"city" + 0.010*"transaction" + 0.008*"analysis"
Topic #5: 0.084*"description" + 0.075*"yet" + 0.020*"weather" + 0.018*"file" + 0.016*"com" + 0.014*"imdb" + 0.014*"kera" + 0.

## LSI Model

**LSI**, also called **LSA (Latent Semantic Analysis)**, is a method that finds **hidden relationships between words and documents** by using **math (linear algebra)** instead of probabilities.

- Each **document** can be represented as a vector of words.  
- Some words mean similar things and often appear together.  
- LSI reduces the data to a smaller set of **concepts (topics)** that capture those hidden patterns.


### How it works

1. Build a **document–term matrix** (counts of each word in each document).  
2. Apply **Singular Value Decomposition (SVD)**, a mathematical technique that breaks the matrix into three smaller matrices:
   - documents × concepts  
   - concepts × words  
3. Keep only the top *k* concepts (topics).  
4. These concepts represent the main themes in the data.


**In short:**  
> LSI uses math (SVD) to uncover hidden themes,  
> showing which documents and words are related through shared meaning —  
> not just exact word matches.


In [36]:
from gensim.models import LsiModel
lsamodel = LsiModel(corpus, num_topics=10, id2word = id2word)
pprint(lsamodel.print_topics(num_topics=10, num_words=10))

[(0,
  '-0.970*"university" + -0.174*"state" + -0.076*"college" + -0.051*"texas" + '
  '-0.049*"california" + -0.039*"institute" + -0.031*"new" + '
  '-0.028*"technology" + -0.027*"florida" + -0.027*"north"'),
 (1,
  '-0.389*"player" + -0.248*"team" + -0.221*"shot" + -0.200*"number" + '
  '-0.177*"time" + -0.173*"file" + -0.159*"year" + -0.156*"csv" + '
  '-0.147*"goal" + -0.126*"ice"'),
 (2,
  '-0.436*"player" + -0.307*"shot" + -0.259*"team" + 0.250*"integer" + '
  '0.225*"strongly" + -0.175*"ice" + -0.173*"goal" + 0.154*"file" + '
  '-0.151*"attempt" + 0.133*"csv"'),
 (3,
  '0.595*"integer" + 0.535*"strongly" + 0.263*"interested" + 0.261*"enjoy" + '
  '0.119*"much" + 0.116*"player" + -0.098*"file" + -0.093*"year" + '
  '0.090*"shot" + -0.088*"csv"'),
 (4,
  '0.402*"year" + -0.325*"date" + -0.266*"element" + -0.199*"tag" + '
  '-0.192*"registration" + -0.186*"zero" + -0.180*"end" + -0.174*"start" + '
  '-0.170*"one" + -0.165*"application"'),
 (5,
  '0.535*"csv" + -0.436*"year" + -0.19

In [37]:
for idx in range(10):
    print("Topic #%s:" % idx, lsamodel.print_topic(idx, 10))
print("=" * 20)

Topic #0: -0.970*"university" + -0.174*"state" + -0.076*"college" + -0.051*"texas" + -0.049*"california" + -0.039*"institute" + -0.031*"new" + -0.028*"technology" + -0.027*"florida" + -0.027*"north"
Topic #1: -0.389*"player" + -0.248*"team" + -0.221*"shot" + -0.200*"number" + -0.177*"time" + -0.173*"file" + -0.159*"year" + -0.156*"csv" + -0.147*"goal" + -0.126*"ice"
Topic #2: -0.436*"player" + -0.307*"shot" + -0.259*"team" + 0.250*"integer" + 0.225*"strongly" + -0.175*"ice" + -0.173*"goal" + 0.154*"file" + -0.151*"attempt" + 0.133*"csv"
Topic #3: 0.595*"integer" + 0.535*"strongly" + 0.263*"interested" + 0.261*"enjoy" + 0.119*"much" + 0.116*"player" + -0.098*"file" + -0.093*"year" + 0.090*"shot" + -0.088*"csv"
Topic #4: 0.402*"year" + -0.325*"date" + -0.266*"element" + -0.199*"tag" + -0.192*"registration" + -0.186*"zero" + -0.180*"end" + -0.174*"start" + -0.170*"one" + -0.165*"application"
Topic #5: 0.535*"csv" + -0.436*"year" + -0.193*"number" + 0.175*"file" + -0.166*"date" + -0.155*"t

## Visualize the topics and documents with the trained Topic Model
- Use pyLDAvis from gensim

In [38]:
%%capture
!pip install pyLDAvis

In [39]:
import pyLDAvis.gensim

### Enable the notebook for visualization

In [40]:
import warnings
warnings.filterwarnings("ignore")

In [41]:
pyLDAvis.enable_notebook()

### Visualize the Topic model

### What is λ (Lambda) in pyLDAvis?

The **λ slider** controls how the most relevant words for each topic are chosen.

- **λ = 1.0** → shows the most **frequent** words in that topic.  
- **λ = 0.0** → shows the most **unique or exclusive** words to that topic.  
- **λ ≈ 0.6 (default)** → balances between frequency and uniqueness.

Move λ **left** to see what makes the topic special,  
move it **right** to see the most common words within the topic.


In [42]:
pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
9     -0.014991  0.114812       1        1  28.089953
13    -0.098947  0.007660       2        1  22.754753
14    -0.083515  0.016020       3        1  10.748526
10    -0.081645  0.088451       4        1   6.505688
4     -0.053103  0.000489       5        1   6.311631
11    -0.085263  0.020073       6        1   3.478501
12    -0.096149 -0.117574       7        1   3.099337
0     -0.031260 -0.327154       8        1   2.947271
3      0.046719  0.107492       9        1   2.873317
6     -0.046443  0.049265      10        1   2.870044
8      0.169490  0.161350      11        1   2.625563
2     -0.042583 -0.019903      12        1   2.272333
1      0.087405  0.076472      13        1   2.012191
5     -0.095490 -0.055992      14        1   1.979258
7      0.425776 -0.121460      15        1   1.431633, topic_info=               Term         Freq        Total Category  logprob  loglift
579      university  1327.000000  1327.000000  Default  30.0000  30.0000
566           state  1032.000000  1032.000000  Default  29.0000  29.0000
603             csv  1232.000000  1232.000000  Default  28.0000  28.0000
772     description   639.000000   639.000000  Default  27.0000  27.0000
291            file  1642.000000  1642.000000  Default  26.0000  26.0000
...             ...          ...          ...      ...      ...      ...
2352           west    13.810431    42.684116  Topic15  -5.4530   3.1180
167   international    19.726899   199.117594  Topic15  -5.0964   1.9344
992          school    18.538806   210.040615  Topic15  -5.1586   1.8189
1135      community    16.169875   271.753310  Topic15  -5.2953   1.4246
373             set    16.299101  1043.429910  Topic15  -5.2873   0.0872

[958 rows x 6 columns], token_table=      Topic      Freq            Term
term                                 
1999      1  0.919922        accident
1999      4  0.071373        accident
1         1  0.562370         account
1        10  0.432592         account
1965      1  0.097653  accountability
...     ...       ...             ...
2919      8  0.926176       zisserman
1793      1  0.339775            zone
1793      5  0.019987            zone
1793     13  0.619590            zone
2888      4  0.900579          zurich

[2300 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[10, 14, 15, 11, 5, 12, 13, 1, 4, 7, 9, 3, 2, 6, 8])

Interpretation of Topic Modeling Visualization (pyLDAvis)

The visualization shows the topics discovered by the LDA model in a two-dimensional space. Each circle represents a topic, and the size of the circle indicates the relative prevalence of that topic in the dataset. Larger circles correspond to more dominant topics.

The distance between circles reflects how similar or different the topics are. Topics that are far apart are more distinct, while overlapping or closely positioned circles indicate that the topics share similar terms and may not be well separated.

On the right side, the most relevant words for the selected topic are displayed. These words help interpret the meaning of each topic. By adjusting the λ (lambda) parameter, we can control whether we emphasize more frequent words (λ closer to 1) or more distinctive words (λ closer to 0).

Overall, the visualization helps us understand the structure of the topics, their importance, and how clearly they are separated from each other. This makes it easier to evaluate the quality of the topic model.
    

### Interpretation of Topics 

The LDA model extracted 10 topics from the dataset. While some topics are interpretable, others appear noisy or less coherent, indicating mixed data quality and preprocessing limitations.

**Topic 0: Universities & Locations**
This topic includes words such as *university, state, college, california, texas*, suggesting references to academic institutions and geographical locations.

**Topic 1: Sports & Game Statistics**
Keywords like *player, team, shot, goal, ice* clearly indicate a sports-related topic, likely hockey or similar team-based games.

**Topic 2: Mixed Sports & Technical Data**
This topic contains both sports terms (*player, team, goal*) and technical terms (*integer, file, csv*), suggesting a mixture of structured data and sports analytics.

**Topic 3: User Preferences / Sentiment**
Words such as *interested, enjoy, strongly* suggest expressions of opinion or sentiment, possibly from user-generated content or reviews.

**Topic 4: Temporal & Structured Data Fields**
This topic includes *year, date, start, end, registration*, indicating structured or tabular data related to time and events.

**Topic 5: CSV & File Data Processing**
Keywords like *csv, file, numeric, text* suggest data processing, possibly related to structured datasets and file formats.

**Topic 6: Numeric & Text Data Representation**
This topic focuses on *numeric, text, language, word*, indicating discussions about data types or textual/numerical processing.

**Topic 7: Machine Learning & Web Data**
Words such as *model, trained, image, http* suggest topics related to machine learning, web data, or model training.

**Topic 8: Multilingual / Noisy Tokens**
This topic includes words like *de, el, en*, which are common in multiple languages (e.g., Spanish, French), indicating multilingual noise or unfiltered tokens.

**Topic 9: Web & Language Noise**
Keywords such as *http, com, de, el* suggest web-related tokens and noise, likely due to insufficient preprocessing (URLs and language artifacts).

---

### Overall Analysis

This topic model shows more noise and less coherence in several topics. While some topics (such as sports and data processing) are meaningful, others are less interpretable due to mixed vocabulary and irrelevant tokens.

The presence of multilingual words and web-related terms (e.g., *http, com, de, el*) indicates that additional preprocessing steps, such as removing URLs and non-English stopwords, could significantly improve model quality.

Overall, this model highlights the importance of thorough text preprocessing in topic modeling. Cleaner input data would likely result in more coherent and interpretable topics.
